# 01 — Tabular data preprocessing: crop-yield regression

## Task 3 — From raw agricultural records to a validated regression table

This notebook is the preprocessing entry point for `notebooks/tabular/task3_crop_yield/`. It applies the workflow described in `docs/tabular_data_preprocessing.md` to the synthetic crop-yield dataset.

The prediction target, `yield_t_ha`, is **numerical and continuous**. This changes how the target is validated: it must be converted to a finite numerical variable and must **not** be discretized into artificial classes during preprocessing.

### Scope

This notebook performs structural preprocessing only:

- load metadata and raw data;
- validate the schema and analytical roles;
- standardize names and data types;
- inspect missing values and duplicates;
- validate binary management indicators;
- validate the continuous numerical target;
- apply broad mechanical range checks;
- export and reload the processed table.

It intentionally does **not** perform target-distribution analysis, correlations with yield, regression diagnostics, transformations, scaling, feature selection, train/test splitting, or modelling.


## 1. Imports

As in the other preprocessing examples, only a minimal scientific Python stack is required. More specialized statistical and visualization libraries belong to later EDA notebooks.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 150)


## 2. Locate the repository and define paths

All paths are derived from the repository root. This keeps the notebook portable across operating systems and user accounts and preserves the raw data unchanged.


In [2]:
def find_project_root(start: Path) -> Path:
    """Return the nearest parent directory containing the expected project folders."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root. Run the notebook from inside the project."
    )

PROJECT_ROOT = find_project_root(Path.cwd())

DATASET_NAME = "synthetic_crop_yield"
RAW_DIR = PROJECT_ROOT / "data" / DATASET_NAME / "raw_data"
PROCESSED_DIR = PROJECT_ROOT / "data" / DATASET_NAME / "processed_data"

RAW_DATA_PATH = RAW_DIR / "synthetic_crop_yield_1200.csv"
METADATA_PATH = RAW_DIR / "metadata.json"
PROCESSED_DATA_PATH = PROCESSED_DIR / "synthetic_crop_yield_1200_processed.csv"
REPORT_PATH = PROCESSED_DIR / "preprocessing_report.json"

print("Repository root:", PROJECT_ROOT)
print("Raw data:", RAW_DATA_PATH.relative_to(PROJECT_ROOT))
print("Metadata:", METADATA_PATH.relative_to(PROJECT_ROOT))
print("Processed output:", PROCESSED_DATA_PATH.relative_to(PROJECT_ROOT))


Repository root: /home/david/Desktop/depto_computacion/ml_2026/demos_machine_learning_2026
Raw data: data/synthetic_crop_yield/raw_data/synthetic_crop_yield_1200.csv
Metadata: data/synthetic_crop_yield/raw_data/metadata.json
Processed output: data/synthetic_crop_yield/processed_data/synthetic_crop_yield_1200_processed.csv


## 3. Read the metadata and verify the task type

The metadata declares a continuous numerical target. This is important because the same column must later be treated differently from a classification label in EDA and modelling.


In [3]:
with METADATA_PATH.open("r", encoding="utf-8") as file:
    metadata = json.load(file)

required_metadata_keys = {
    "label",
    "label_type",
    "col_id",
    "categorical_column",
    "numerical_column",
}
missing_metadata_keys = required_metadata_keys - set(metadata)
if missing_metadata_keys:
    raise KeyError(f"Missing metadata keys: {sorted(missing_metadata_keys)}")

if metadata["label_type"] != "numerical_continuous":
    raise ValueError("Task 3 expects label_type='numerical_continuous'.")

TARGET_COLUMN = metadata["label"]
ID_COLUMN = metadata["col_id"]
CATEGORICAL_COLUMNS = list(metadata["categorical_column"])
NUMERICAL_COLUMNS = list(metadata["numerical_column"])

print("Identifier column:", ID_COLUMN)
print("Target column:", TARGET_COLUMN)
print("Label type:", metadata["label_type"])
print("Categorical predictors:", len(CATEGORICAL_COLUMNS))
print("Numerical predictors:", len(NUMERICAL_COLUMNS))


Identifier column: plot_id
Target column: yield_t_ha
Label type: numerical_continuous
Categorical predictors: 6
Numerical predictors: 13


## 4. Read the raw table

Common textual missing-value markers are converted to actual missing values at import time. This prevents accidental treatment of placeholders as valid measurements or categories.


In [4]:
MISSING_TOKENS = [
    "NA",
    "N/A",
    "null",
    "None",
    "missing",
    "unknown",
]

raw_df = pd.read_csv(
    RAW_DATA_PATH,
    na_values=MISSING_TOKENS,
    keep_default_na=True,
)

print(f"Loaded {raw_df.shape[0]} rows and {raw_df.shape[1]} columns.")
display(raw_df.head())


Loaded 1200 rows and 21 columns.


,plot_id,mean_temperature_c,season_rainfall_mm,soil_ph,soil_organic_matter_pct,nitrogen_kg_ha,phosphorus_kg_ha,potassium_kg_ha,irrigation_mm,solar_radiation_mj_m2_day,field_slope_pct,planting_density_plants_m2,pest_pressure_score_0_10,soil_moisture_pct,irrigation_system_drip,cultivar_improved,previous_legume_crop,organic_management,fungicide_used,drought_event,yield_t_ha
0,CP0001,20.7,273.7,7.21,3.80,136.6,57.4,38.5,88.3,15.4,6.5,7.93,2.7,28.2,0,1,0,0,1,0,5.38
1,CP0002,24.6,237.9,7.10,2.76,155.4,56.3,113.9,270.5,15.4,1.9,8.38,6.3,27.7,0,1,0,0,1,1,3.93
2,CP0003,20.4,192.5,7.16,2.95,121.6,51.9,49.6,62.2,14.6,4.7,9.82,0.7,21.2,1,0,0,0,0,1,3.10
3,CP0004,24.6,599.3,6.60,4.79,182.7,81.5,97.3,137.4,19.7,1.8,7.35,3.4,33.3,0,1,0,0,0,0,6.61
4,CP0005,11.0,603.6,6.62,2.97,122.8,47.8,97.2,76.1,15.8,9.4,5.94,7.2,35.0,1,0,0,0,1,0,0.70


## 5. Validate the schema against metadata

The raw table must contain the identifier, all declared predictors, and the continuous target—no more and no less for this teaching example.


In [5]:
expected_columns_raw = {
    ID_COLUMN,
    TARGET_COLUMN,
    *CATEGORICAL_COLUMNS,
    *NUMERICAL_COLUMNS,
}
observed_columns = set(raw_df.columns)

missing_columns = sorted(expected_columns_raw - observed_columns)
unexpected_columns = sorted(observed_columns - expected_columns_raw)

display(pd.DataFrame({
    "check": ["Missing expected columns", "Unexpected columns"],
    "result": [missing_columns or "None", unexpected_columns or "None"],
}))

if missing_columns or unexpected_columns:
    raise ValueError(
        "The input schema does not match metadata. "
        f"Missing={missing_columns}, unexpected={unexpected_columns}"
    )


,check,result
0,Missing expected columns,None
1,Unexpected columns,None


## 6. Make analytical roles explicit

The regression target is numerical, but it is **not** an ordinary numerical predictor. Keeping target and predictors as separate roles is essential for later leakage-safe modelling.


In [6]:
def column_role(column: str) -> str:
    if column == ID_COLUMN:
        return "identifier"
    if column == TARGET_COLUMN:
        return "target_numerical_continuous"
    if column in CATEGORICAL_COLUMNS:
        return "categorical_predictor"
    if column in NUMERICAL_COLUMNS:
        return "numerical_predictor"
    return "unassigned"

role_table = pd.DataFrame({
    "column": raw_df.columns,
    "role": [column_role(column) for column in raw_df.columns],
    "raw_dtype": [str(raw_df[column].dtype) for column in raw_df.columns],
})
display(role_table)


,column,role,raw_dtype
0,plot_id,identifier,str
1,mean_temperature_c,numerical_predictor,float64
2,season_rainfall_mm,numerical_predictor,float64
3,soil_ph,numerical_predictor,float64
4,soil_organic_matter_pct,numerical_predictor,float64
5,nitrogen_kg_ha,numerical_predictor,float64
6,phosphorus_kg_ha,numerical_predictor,float64
7,potassium_kg_ha,numerical_predictor,float64
8,irrigation_mm,numerical_predictor,float64
9,solar_radiation_mj_m2_day,numerical_predictor,float64


## 7. Standardize column names

Names are normalized conservatively and the metadata-derived role lists are synchronized with the normalized schema.


In [7]:
def normalize_column_name(name: str) -> str:
    return (
        str(name)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

normalized_columns = [normalize_column_name(column) for column in raw_df.columns]
if len(normalized_columns) != len(set(normalized_columns)):
    raise ValueError("Column-name normalization would create duplicated column names.")

rename_map = dict(zip(raw_df.columns, normalized_columns))
changed_names = {old: new for old, new in rename_map.items() if old != new}
print("Column names changed:", changed_names or "None")

df = raw_df.rename(columns=rename_map).copy()
ID_COLUMN = normalize_column_name(ID_COLUMN)
TARGET_COLUMN = normalize_column_name(TARGET_COLUMN)
CATEGORICAL_COLUMNS = [normalize_column_name(c) for c in CATEGORICAL_COLUMNS]
NUMERICAL_COLUMNS = [normalize_column_name(c) for c in NUMERICAL_COLUMNS]
FINAL_COLUMN_ORDER = normalized_columns
EXPECTED_COLUMNS = {ID_COLUMN, TARGET_COLUMN, *CATEGORICAL_COLUMNS, *NUMERICAL_COLUMNS}


Column names changed: None


## 8. Convert predictors and target according to their roles

For regression, both numerical predictors and the target must be convertible to numerical values. The target is handled separately so its semantic role remains explicit.


In [8]:
conversion_failures = {}

df[ID_COLUMN] = df[ID_COLUMN].astype("string").str.strip()

for column in NUMERICAL_COLUMNS:
    original_non_missing = df[column].notna()
    converted = pd.to_numeric(df[column], errors="coerce")
    failed = original_non_missing & converted.isna()
    conversion_failures[column] = int(failed.sum())
    df[column] = converted

for column in CATEGORICAL_COLUMNS:
    original_non_missing = df[column].notna()
    converted = pd.to_numeric(df[column], errors="coerce")
    failed = original_non_missing & converted.isna()
    conversion_failures[column] = int(failed.sum())
    df[column] = converted.astype("Int64")

original_target_non_missing = df[TARGET_COLUMN].notna()
converted_target = pd.to_numeric(df[TARGET_COLUMN], errors="coerce")
target_conversion_failed = original_target_non_missing & converted_target.isna()
conversion_failures[TARGET_COLUMN] = int(target_conversion_failed.sum())
df[TARGET_COLUMN] = converted_target

failure_table = pd.DataFrame({
    "column": list(conversion_failures),
    "conversion_failures": list(conversion_failures.values()),
})
failure_table = failure_table[failure_table["conversion_failures"] > 0]
display(failure_table if not failure_table.empty else pd.DataFrame({"conversion_failures": ["None"]}))

if any(conversion_failures.values()):
    raise ValueError("At least one value could not be converted to its expected data type.")


,conversion_failures
0,None


## 9. Check missing values

We identify missing values but do not impute them here. Leakage-safe imputation, if required, should later be learned from training data rather than from the complete dataset.


In [9]:
missing_summary = df.isna().sum().rename("missing_values").to_frame()
missing_summary = missing_summary[missing_summary["missing_values"] > 0]
display(missing_summary if not missing_summary.empty else pd.DataFrame({"missing_values": ["None"]}))


,missing_values
0,None


## 10. Check exact duplicates and repeated plot identifiers

A duplicated full row can be removed mechanically. Multiple non-identical rows sharing a plot identifier are ambiguous and therefore trigger review instead of being silently merged.


In [10]:
exact_duplicate_count = int(df.duplicated().sum())
duplicated_id_count = int(df[ID_COLUMN].duplicated().sum())

print("Exact duplicated rows:", exact_duplicate_count)
print("Duplicated identifiers:", duplicated_id_count)

if exact_duplicate_count > 0:
    df = df.drop_duplicates().copy()
    print(f"Removed {exact_duplicate_count} exact duplicated rows.")

remaining_duplicated_ids = int(df[ID_COLUMN].duplicated().sum())
if remaining_duplicated_ids > 0:
    display(df.loc[df[ID_COLUMN].duplicated(keep=False), [ID_COLUMN]])
    raise ValueError("Repeated plot identifiers remain after exact deduplication.")


Exact duplicated rows: 0
Duplicated identifiers: 0


## 11. Validate binary categorical predictors

Management and event indicators are encoded as `0/1`. This step verifies the encoding only; differences in yield between these groups belong to the future regression EDA.


In [11]:
ALLOWED_BINARY_VALUES = {0, 1}
invalid_binary_values = {}

for column in CATEGORICAL_COLUMNS:
    observed = set(df[column].dropna().astype(int).unique().tolist())
    invalid = sorted(observed - ALLOWED_BINARY_VALUES)
    if invalid:
        invalid_binary_values[column] = invalid

print("Invalid binary encodings:", invalid_binary_values or "None")
if invalid_binary_values:
    raise ValueError(f"Invalid binary encodings found: {invalid_binary_values}")


Invalid binary encodings: None


## 12. Validate the continuous target

A regression target must be numerical and finite. We do not summarize its distribution or create arbitrary bins here, because both operations belong to exploratory analysis rather than structural preprocessing.


In [12]:
target_values = df[TARGET_COLUMN].dropna()
target_non_finite = int((~np.isfinite(target_values)).sum())
target_non_positive = int((target_values <= 0).sum())

continuous_target_validation = pd.DataFrame({
    "check": ["Missing target values", "Non-finite target values", "Non-positive target values"],
    "count": [int(df[TARGET_COLUMN].isna().sum()), target_non_finite, target_non_positive],
})
display(continuous_target_validation)

if int(df[TARGET_COLUMN].isna().sum()) or target_non_finite or target_non_positive:
    raise ValueError("The continuous target failed structural validation.")


,check,count
0,Missing target values,0
1,Non-finite target values,0
2,Non-positive target values,0


## 13. Apply broad numerical sanity checks

These checks detect impossible or clearly invalid encodings. They are intentionally broad: unusual but possible observations should remain for later outlier analysis rather than being removed during preprocessing.


In [13]:
invalid_numeric_checks = {}

for column in NUMERICAL_COLUMNS:
    values = df[column].dropna()
    non_finite_count = int((~np.isfinite(values)).sum())
    if non_finite_count:
        invalid_numeric_checks[f"{column}: non-finite"] = non_finite_count

constraints = {
    "season_rainfall_mm": lambda s: (s < 0),
    "soil_ph": lambda s: (s < 0) | (s > 14),
    "soil_organic_matter_pct": lambda s: (s < 0) | (s > 100),
    "nitrogen_kg_ha": lambda s: (s < 0),
    "phosphorus_kg_ha": lambda s: (s < 0),
    "potassium_kg_ha": lambda s: (s < 0),
    "irrigation_mm": lambda s: (s < 0),
    "solar_radiation_mj_m2_day": lambda s: (s < 0),
    "field_slope_pct": lambda s: (s < 0),
    "planting_density_plants_m2": lambda s: (s <= 0),
    "pest_pressure_score_0_10": lambda s: (s < 0) | (s > 10),
    "soil_moisture_pct": lambda s: (s < 0) | (s > 100),
}

for column, invalid_rule in constraints.items():
    invalid_count = int(invalid_rule(df[column].dropna()).sum())
    if invalid_count:
        invalid_numeric_checks[column] = invalid_count

print("Invalid numerical values:", invalid_numeric_checks or "None")
if invalid_numeric_checks:
    raise ValueError(f"Numerical sanity checks failed: {invalid_numeric_checks}")


Invalid numerical values: None


## 14. Final structural validation

The final table must preserve its schema, unique identifiers, finite continuous target, valid categorical encodings, and original column order.


In [14]:
df = df.loc[:, FINAL_COLUMN_ORDER].reset_index(drop=True)

assert list(df.columns) == FINAL_COLUMN_ORDER, "Column order changed unexpectedly."
assert set(df.columns) == EXPECTED_COLUMNS, "Final schema differs from metadata."
assert df[ID_COLUMN].notna().all(), "Missing identifiers remain."
assert df[ID_COLUMN].is_unique, "Identifiers are not unique."
assert df[TARGET_COLUMN].notna().all(), "Missing target values remain."
assert np.isfinite(df[TARGET_COLUMN]).all(), "Non-finite target values remain."
assert df.duplicated().sum() == 0, "Exact duplicate rows remain."
assert not any(conversion_failures.values()), "Data-type conversion failures remain."
assert not invalid_binary_values, "Invalid categorical encodings remain."
assert not invalid_numeric_checks, "Invalid numerical values remain."

final_validation = pd.DataFrame({
    "property": ["Rows", "Columns", "Exact duplicate rows", "Duplicated identifiers", "Missing values", "Target dtype"],
    "value": [
        df.shape[0], df.shape[1], int(df.duplicated().sum()), int(df[ID_COLUMN].duplicated().sum()),
        int(df.isna().sum().sum()), str(df[TARGET_COLUMN].dtype),
    ],
})
display(final_validation)


,property,value
0,Rows,1200
1,Columns,21
2,Exact duplicate rows,0
3,Duplicated identifiers,0
4,Missing values,0
5,Target dtype,float64


## 15. Export the processed dataset and preprocessing report

The report records the analytical roles and validation outcome so the later regression EDA can load a stable, machine-readable contract instead of re-deriving column roles.


In [15]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_DATA_PATH, index=False)

preprocessing_report = {
    "dataset": DATASET_NAME,
    "task_type": "regression",
    "label_type": metadata["label_type"],
    "input_file": str(RAW_DATA_PATH.relative_to(PROJECT_ROOT)),
    "metadata_file": str(METADATA_PATH.relative_to(PROJECT_ROOT)),
    "output_file": str(PROCESSED_DATA_PATH.relative_to(PROJECT_ROOT)),
    "input_shape": [int(raw_df.shape[0]), int(raw_df.shape[1])],
    "output_shape": [int(df.shape[0]), int(df.shape[1])],
    "identifier_column": ID_COLUMN,
    "target_column": TARGET_COLUMN,
    "numerical_columns": NUMERICAL_COLUMNS,
    "categorical_columns": CATEGORICAL_COLUMNS,
    "exact_duplicate_rows_found": exact_duplicate_count,
    "duplicated_identifiers_found": duplicated_id_count,
    "missing_values_after_preprocessing": int(df.isna().sum().sum()),
    "target_non_finite_values": target_non_finite,
}

with REPORT_PATH.open("w", encoding="utf-8") as file:
    json.dump(preprocessing_report, file, indent=2)

print("Processed dataset saved to:", PROCESSED_DATA_PATH.relative_to(PROJECT_ROOT))
print("Preprocessing report saved to:", REPORT_PATH.relative_to(PROJECT_ROOT))


Processed dataset saved to: data/synthetic_crop_yield/processed_data/synthetic_crop_yield_1200_processed.csv
Preprocessing report saved to: data/synthetic_crop_yield/processed_data/preprocessing_report.json


## 16. Reload the exported table

The exported CSV is reloaded and compared against the in-memory processed DataFrame. This round-trip check catches serialization problems immediately.


In [16]:
reloaded_df = pd.read_csv(PROCESSED_DATA_PATH)
pd.testing.assert_frame_equal(
    df.reset_index(drop=True),
    reloaded_df.reset_index(drop=True),
    check_dtype=False,
)
print("Round-trip validation: PASS")
print("Reloaded shape:", reloaded_df.shape)


Round-trip validation: PASS
Reloaded shape: (1200, 21)


## 17. What was intentionally not done here?

This notebook stops at a validated regression table. It does **not** inspect the shape of the yield distribution, correlations with the target, nonlinear relationships, conditional target distributions by categorical variables, heteroscedasticity, PCA, outliers in a statistical sense, or regression models.

Those operations will form the dedicated **EDA for a numerical label**. Keeping them separate from preprocessing makes the learning objectives and the data pipeline much clearer.
